## Work with GPU

In [ ]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

In [ ]:
torch.cuda.is_available()

In [ ]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device being used:',device)

In [ ]:
device

In [ ]:
df = pd.read_csv("fashion-mnist_train.csv")
df.shape

In [ ]:
df.head()

In [ ]:
features = df.iloc[:,1:]
targets = df.iloc[:,0]
targets

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(features, targets, test_size=0.2, random_state= 42)
x_train /= 255.0
x_test /= 255.0

In [ ]:
x_train.shape

In [ ]:
x_test.shape

In [ ]:
device

In [ ]:
# to create the dataset
class CustomDataset(Dataset):
    def __init__(self, features, targets):
        self.features = torch.tensor(features.to_numpy()).to(torch.float32)
        self.targets = torch.tensor(targets.to_numpy()).to(torch.long)
    
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        return (self.features[index], self.targets[index])

In [ ]:
# dataset collection stored here
train_dataset = CustomDataset(x_train, y_train)
test_dataset = CustomDataset(x_test, y_test)

In [ ]:
train_dataset.__len__()

In [ ]:
test_dataset.__len__()

In [ ]:
train_dataset.__getitem__(0)

In [ ]:
train_dataset_loader = DataLoader(train_dataset, batch_size= 32, shuffle= True, pin_memory=True)
test_dataset_loader = DataLoader(test_dataset, batch_size= 32, shuffle=False, pin_memory=True)

In [ ]:
for feature, target in train_dataset_loader:
    print(feature)
    print(target)
    break

In [40]:
class ImageClassifier(nn.Module):

    def __init__(self, number_of_features):
        super().__init__()

        # Image classifier model
        self.model = nn.Sequential(
            nn.Linear(number_of_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p = 0.4),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p = 0.4),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(p = 0.4),
            nn.Linear(32, 10)
        )

        # optimizer
        self.optim = optim.SGD(self.model.parameters(), lr = 0.1, weight_decay= 1e-4)
    
    def forward(self, features):
        self.y_pred = self.model(features)
        return self.y_pred
    
    def loss_function(self, y_real):
        loss_fn = nn.CrossEntropyLoss()
        self.loss = loss_fn(self.y_pred, y_real)
        return self.loss.item()
    
    def optimization(self):
        self.optim.zero_grad()
        self.loss.backward()
        self.optim.step()


In [41]:
# class ImageClassifier(nn.Module):

#     def __init__(self, number_of_features):
#         super().__init__()

#         # Image classifier model
#         self.model = nn.Sequential(
#             nn.Linear(number_of_features, 128),
#             # nn.BatchNorm1d(128),
#             nn.ReLU(),
#             # nn.Dropout(p = 0.4),
#             nn.Linear(128, 64),
#             # nn.BatchNorm1d(64),
#             nn.ReLU(),
#             # nn.Dropout(p = 0.4),
#             nn.Linear(64, 32),
#             # nn.BatchNorm1d(32),
#             nn.ReLU(),
#             # nn.Dropout(p = 0.4),
#             nn.Linear(32, 10)
#         )

#         # optimizer
#         self.optim = optim.SGD(self.model.parameters(), lr = 0.1, weight_decay= 1e-4)
    
#     def forward(self, features):
#         self.y_pred = self.model(features)
#         return self.y_pred
    
#     def loss_function(self, y_real):
#         loss_fn = nn.CrossEntropyLoss()
#         self.loss = loss_fn(self.y_pred, y_real)
#         return self.loss.item()
    
#     def optimization(self):
#         self.optim.zero_grad()
#         self.loss.backward()
#         self.optim.step()


In [42]:
x_train.shape[1]

784

In [43]:
# model is defined but the model uses the "cpu"
model = ImageClassifier(x_train.shape[1])

In [44]:
for i in model.named_parameters():
    print(i)

('model.0.weight', Parameter containing:
tensor([[-0.0191, -0.0342,  0.0009,  ...,  0.0113,  0.0344, -0.0172],
        [ 0.0281, -0.0159,  0.0112,  ...,  0.0233, -0.0039, -0.0162],
        [ 0.0005, -0.0281, -0.0002,  ..., -0.0208,  0.0266, -0.0207],
        ...,
        [-0.0226,  0.0232, -0.0185,  ...,  0.0329, -0.0349, -0.0055],
        [-0.0292, -0.0056, -0.0050,  ..., -0.0207, -0.0306, -0.0274],
        [ 0.0061,  0.0111,  0.0317,  ...,  0.0023, -0.0225,  0.0179]],
       requires_grad=True))
('model.0.bias', Parameter containing:
tensor([-0.0058, -0.0164, -0.0236, -0.0111, -0.0113,  0.0183, -0.0218,  0.0262,
        -0.0289,  0.0172, -0.0192, -0.0179,  0.0291, -0.0305,  0.0167, -0.0238,
        -0.0042, -0.0356, -0.0166,  0.0064, -0.0229,  0.0046, -0.0017, -0.0256,
         0.0021,  0.0018, -0.0012,  0.0031,  0.0294, -0.0165,  0.0015,  0.0037,
         0.0076, -0.0076, -0.0128, -0.0057,  0.0283,  0.0032,  0.0216,  0.0050,
         0.0194,  0.0016, -0.0284,  0.0303, -0.0091, -0.03

In [45]:
device

device(type='cuda')

In [46]:
# making model use gpu instead of cpu
model = model.to(device)

In [47]:
model

ImageClassifier(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.4, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.4, inplace=False)
    (12): Linear(in_features=32, out_features=10, bias=True)
  )
)

In [48]:
epochs = 30
total_loss = 0
for epoch in range(epochs):
    epoch += 1
    total_loss = 0
    for features_batch, targets_batch in train_dataset_loader:
        features_batch = features_batch.to(device)
        targets_batch = targets_batch.to(device)
        y_pred = model(features_batch)
        loss = model.loss_function(targets_batch)
        model.optimization()
        total_loss = total_loss + loss

    print("epoch:", epoch, "\tloss:", total_loss/len(train_dataset_loader))

epoch: 1 	loss: 0.8724587519367536
epoch: 2 	loss: 0.6791071786681812
epoch: 3 	loss: 0.6396959502895673
epoch: 4 	loss: 0.6142685723602772
epoch: 5 	loss: 0.5864119452436765
epoch: 6 	loss: 0.5712723227639993
epoch: 7 	loss: 0.5599117186466853
epoch: 8 	loss: 0.5543900786042213
epoch: 9 	loss: 0.5370111434062322
epoch: 10 	loss: 0.531536076982816
epoch: 11 	loss: 0.5285744131207466
epoch: 12 	loss: 0.5252895933190982
epoch: 13 	loss: 0.518468517596523
epoch: 14 	loss: 0.5140389748166004
epoch: 15 	loss: 0.5104959758917491
epoch: 16 	loss: 0.5034543517033259
epoch: 17 	loss: 0.5000474280615648
epoch: 18 	loss: 0.4990306584040324
epoch: 19 	loss: 0.49596707106630006
epoch: 20 	loss: 0.49427720956504345
epoch: 21 	loss: 0.49354988512396814
epoch: 22 	loss: 0.48509836245576543
epoch: 23 	loss: 0.48961357523997623
epoch: 24 	loss: 0.47764414248863857
epoch: 25 	loss: 0.48242153961459794
epoch: 26 	loss: 0.47801352460185687
epoch: 27 	loss: 0.4741764973104
epoch: 28 	loss: 0.471734929655989

In [53]:
# model ready for evaluation
model.eval()

ImageClassifier(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.4, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.4, inplace=False)
    (12): Linear(in_features=32, out_features=10, bias=True)
  )
)

### Accuracy:
We need to find the accurcy now. The accuracy of the data is given by the formula:

Accuracy = (number of correct predictions) / (total number of the predictions)

In [54]:
len(test_dataset_loader)*32

12000

In [55]:
correct = 0
data_records = 0
with torch.no_grad():
    for features_batch, targets_batch in test_dataset_loader:
        features_batch = features_batch.to(device)
        targets_batch = targets_batch.to(device)
        y_pred = model(features_batch)
        correct = correct + (targets_batch == torch.max(y_pred, dim = 1)[1]).sum().item()
        data_records = data_records + features_batch.shape[0]
    print("total data_records:", data_records, "correct records:", correct)
    accuracy = correct / data_records
    print(accuracy)

total data_records: 12000 correct records: 10560
0.88


In [56]:
correct = 0
data_records = 0
with torch.no_grad():
    for features_batch, targets_batch in train_dataset_loader:
        features_batch = features_batch.to(device)
        targets_batch = targets_batch.to(device)
        y_pred = model(features_batch)
        correct = correct + (targets_batch == torch.max(y_pred, dim = 1)[1]).sum().item()
        data_records = data_records + features_batch.shape[0]
    print("total data_records:", data_records, "correct records:", correct)
    accuracy = correct / data_records
    print(accuracy)

total data_records: 48000 correct records: 43216
0.9003333333333333


## Hyperparameter Tuning using Optuna:

In [ ]:
# defining the class for ImgaeClassifier but in a different way:
class ImageClassifier(nn.Module):

    def __init__(self, number_of_features):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(number_of_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p = 0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p = 0.3),
            nn.Linear(64,32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(p = 0.3),
            nn.Linear(32, 10)
        )

        def forward(self, features):
            self.y_pred = self.model(features)
            return self.y_pred

In [ ]:
lr = 0.01
epochs = 30

In [ ]:
x_train.shape[1]

In [ ]:
model = ImageClassifier(x_train.shape[1])

model = model.to(device)

In [ ]:
# loss function
loss_fn = nn.CrossEntropyLoss()

# optimization
optim = torch.optim.SGD(model.parameters(), lr, weight_decay= 1e-4)